## Calculate the probability desity function $P(\widetilde{X})$ in the SOI

$$
R_{SOI} = a(\frac{m}{M})^{2/5}
$$

In [3]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from utils.Utils import (
    CanonicalUnits, 
    GravitationalParameters, 
    trasformation_X_to_E, 
    compute_jacobian_XoE,
    trasformation_E_to_X, 
    computeNumericalJacobian,
)
import numpy as np
import multimin as mm
from tqdm import tqdm
import plotly.graph_objects as go


In [4]:
deg = np.pi/180
AU_m = 1.496e11 #m
M_sun = 1.9891e30
G = 6.67430e-11 # m^3 / (kg s^2)
year = 365.25*24*3600 #s
mu = CanonicalUnits().mu
grav_params = GravitationalParameters(mu=mu)

In [5]:
F = mm.FitCMND(f"../../multimin/products/fit-NEAs-qei-Ng10-Nv3-rad.pkl")

In [6]:
def P_E_CMND(a: float, e: float, i: float, F: mm.FitCMND) -> float:
    max = 2*np.pi; min = 0

    q = a*(1-e)
    element = np.array([q, e, i])
    scales=[1.35,1.00,np.pi]
    u_element = mm.Util.tIF(element, scales, mm.Util.f2u)

    P_qei = F.cmnd.pdf(u_element)
    P_WwM = 1/(max - min)**3

    return P_qei * P_WwM

def Jacobian_qei_to_QEI(q: float, e: float, i: float, q_max: float, e_max: float, i_max: float) -> np.array:
    partialA_a = q_max/(q*(q_max - q))
    partialA_e = 0
    partialA_i = 0
    partialE_a = 0
    partialE_e = e_max/(e*(e_max - e))
    partialE_i = 0
    partialI_a = 0
    partialI_e = 0
    partialI_i = i_max/(i*(i_max - i))

    Jacobian = np.array([
        [partialA_a, partialA_e, partialA_i], 
        [partialE_a, partialE_e, partialE_i], 
        [partialI_a, partialI_e, partialI_i]
        ])
        
    return Jacobian

def P_X_CMND(x: np.array, y: np.array, z: np.array, vx: np.array, vy: np.array, vz: np.array, q_max: float, e_max: float, i_max: float, mu: float, F: mm.FitCMND) -> np.array:
    """
    Vectorized version: x, y, vx, vy are arrays (or scalars).
    Returns array of P values.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    z = np.asarray(z)
    vx = np.asarray(vx)
    vy = np.asarray(vy)
    vz = np.asarray(vz)
    # Prepare output array
    shape = np.broadcast(x, y, z, vx, vy, vz).shape
    P = np.empty(shape, dtype=float)

    # Flatten for iteration if needed
    x_flat = x.ravel()
    y_flat = y.ravel()
    z_flat = z.ravel()
    vx_flat = vx.ravel()
    vy_flat = vy.ravel()
    vz_flat = vz.ravel()

    neas_number = 0
    not_neas_number = 0

    for idx in range(x_flat.size):
        a, e, i, Omega, w, M = trasformation_X_to_E(x_flat[idx], y_flat[idx], z_flat[idx], vx_flat[idx], vy_flat[idx], vz_flat[idx], mu)
        q = a*(1-e)
        if q > 1.3 or e > 1: 
            P.flat[idx] = 0
            neas_number += 1
        else: 
            J = compute_jacobian_XoE(a,e,i,Omega,w,M,mu)
            J_qei_to_QEI = Jacobian_qei_to_QEI(q,e,i,q_max,e_max,i_max)
            with np.errstate(divide='ignore', invalid='ignore'):
                det = np.linalg.det(J)
                det_QEI = np.linalg.det(J_qei_to_QEI)
                if det == 0 or not np.isfinite(det):
                    print("Problematic point: ", x_flat[idx], y_flat[idx], z_flat[idx], vx_flat[idx], vy_flat[idx], vz_flat[idx])
                    P.flat[idx] = np.nan
                else:
                    inv_det = 1.0/det 
                    P.flat[idx] = P_E_CMND(a, e, i, F) * abs(inv_det) * abs(det_QEI)
                    not_neas_number += 1

                    if np.isnan(P_E_CMND(a, e, i, F) * abs(inv_det) * abs(det_QEI)):
                        print('Nan value encounter: ', P_E_CMND(a, e, i, F), abs(inv_det), abs(det_QEI))
                        print("Problematic point: ", x_flat[idx], y_flat[idx], z_flat[idx], vx_flat[idx], vy_flat[idx], vz_flat[idx])
                    
    return P.reshape(shape)

def sphere_surface_integral_P_X_CMND(
    c: np.array,
    r: float,
    v: np.array,
    dv: float,
    deg: int,
    q_max: float,
    e_max: float,
    i_max: float,
    mu: float,
    F: object
) -> float:
    """
    Calculates the 6D integral of P_X_CMND using Gauss-Legendre Quadrature.
    Robust to P_X_CMND returning extra diagnostic values.
    """
    from numpy.polynomial.legendre import leggauss
    # 1. Get Gauss-Legendre nodes and weights
    nodes, weights = leggauss(deg)

    # 2. Velocity Mesh (Cartesian Box)
    vx_vals = v[0] + dv * nodes
    vy_vals = v[1] + dv * nodes
    vz_vals = v[2] + dv * nodes
    
    # 3. Position Mesh (Spherical Coordinates)
    # rho: [0, r], theta: [0, pi], phi: [0, 2pi]
    rho_vals = (r / 2.0) * (nodes + 1)
    theta_vals = (np.pi / 2.0) * (nodes + 1)
    phi_vals = np.pi * (nodes + 1)

    # 4. Construct the 6D Meshgrid
    # Order: rho, theta, phi, vx, vy, vz
    Rho, Theta, Phi, Vx, Vy, Vz = np.meshgrid(
        rho_vals, theta_vals, phi_vals, vx_vals, vy_vals, vz_vals, indexing='ij'
    )
    
    # Mesh the weights
    W_rho, W_theta, W_phi, W_vx, W_vy, W_vz = np.meshgrid(
        weights, weights, weights, weights, weights, weights, indexing='ij'
    )

    # 5. Coordinate Transformations
    sin_theta = np.sin(Theta)
    cos_theta = np.cos(Theta)
    sin_phi = np.sin(Phi)
    cos_phi = np.cos(Phi)
    
    X = c[0] + Rho * sin_theta * cos_phi
    Y = c[1] + Rho * sin_theta * sin_phi
    Z = c[2] + Rho * cos_theta

    # 6. Calculate Integration Weights (Jacobians)
    scale_factors = (r / 2.0) * (np.pi / 2.0) * np.pi * (dv**3)
    spherical_jacobian = (Rho**2) * sin_theta
    
    total_weights = (
        W_rho * W_theta * W_phi * W_vx * W_vy * W_vz 
        * spherical_jacobian 
        * scale_factors
    )

    # 7. Evaluate Function
    P_output = P_X_CMND(
        X.ravel(), Y.ravel(), Z.ravel(), 
        Vx.ravel(), Vy.ravel(), Vz.ravel(),
        q_max, e_max, i_max, mu, F
    )
    
    # --- FIX FOR TUPLE RETURN ---
    # If P_X_CMND returns (P, neas_count, etc.), take the first element.
    if isinstance(P_output, tuple):
        P_flat = P_output[0]
    else:
        P_flat = P_output
        
    # Ensure it is a numpy array
    P_flat = np.asarray(P_flat)
    # ----------------------------

    # 8. Integrate
    # Reshape P_flat to match the 6D grid shape of total_weights
    try:
        P_grid = P_flat.reshape(total_weights.shape)
    except ValueError as e:
        print(f"Shape Error Details: P_flat size {P_flat.size}, Weights shape {total_weights.shape}")
        raise e
    
    weighted_values = P_grid * total_weights
    
    integral_result = np.nansum(weighted_values)
    
    return integral_result

In [7]:
a = 1.0
m = 5.972e24/M_sun
M = 1.0
R_SOI = a*(m/M)**(2/5)
R_SOI

0.006179954429750629

In [14]:
a = 1.0
m = 5.972e24/M_sun
M = 1.0
R_SOI = a*(m/M)**(2/5)

c_x, c_y, c_z = 1.0, 0.0, 0.0
center_pos = np.array([c_x, c_y, c_z])

v_x, v_y, v_z = 0.0, (mu/1)**0.5, 0.0
center_vel = np.array([v_x, v_y, v_z])
dvxyz = 5000 * (1/AU_m) * year 

In [16]:
dvxyz

1.0547326203208556

In [9]:
a = 1.0
m = 5.972e24/M_sun
M = 1.0
R_SOI = a*(m/M)**(2/5)

c_x, c_y, c_z = 1.0, 0.0, 0.0
center_pos = np.array([c_x, c_y, c_z])

v_x, v_y, v_z = 0.0, (mu/1)**0.5, 0.0
center_vel = np.array([v_x, v_y, v_z])
dvxyz = 5000 * (1/AU_m) * year 

degree = 6 
max_elements = (1.3, 1.0, np.pi) 

result = sphere_surface_integral_P_X_CMND(
    center_pos, R_SOI, 
    center_vel, dvxyz, 
    degree, 
    q_max=max_elements[0], e_max=max_elements[1], i_max=max_elements[2], mu=mu, F=F
)

print("Integral Result:", result)

Integral Result: 1.316778034162785e-07


In [10]:
PX_radi = []
radios = np.linspace(R_SOI, 0.5, 20)

for r in radios:
    v_x, v_y, v_z = 0.0, (mu/1)**0.5, 0.0
    center_vel = np.array([v_x, v_y, v_z])
    result = sphere_surface_integral_P_X_CMND(
        center_pos, r, 
        center_vel, dvxyz, 
        degree, 
        q_max=max_elements[0], e_max=max_elements[1], i_max=max_elements[2], mu=mu, F=F
    )
    PX_radi.append(result)


In [11]:
PX_radi 

[np.float64(1.316778034162785e-07),
 np.float64(1.7592480111419256e-05),
 np.float64(9.444114354812087e-05),
 np.float64(0.00025569797025653694),
 np.float64(0.000509540061595542),
 np.float64(0.0008470460716552717),
 np.float64(0.0012465169293393195),
 np.float64(0.0017121309532123312),
 np.float64(0.0022177906244624382),
 np.float64(0.002725804060984429),
 np.float64(0.003268937228391722),
 np.float64(0.0038094442957602014),
 np.float64(0.0043965810784174645),
 np.float64(0.005068308486331787),
 np.float64(0.005495418242679341),
 np.float64(0.006234670318265515),
 np.float64(0.00623529676043242),
 np.float64(0.006741875039421641),
 np.float64(0.007325195574869282),
 np.float64(0.008053866946509461)]

In [ ]:
import plotly.graph_objects as go


fig = go.Figure()

fig.add_trace(go.Scatter(x=radios, y=PX_radi,
                    mode='lines+markers',
                    name='lines+markers'))
fig.update_layout(
    title="Probability Density Function and Radius from the Earth",  
    xaxis_title="Radius from Earth",     
    yaxis_title="Probability Density Function"    
)
fig.show()

In [12]:
import numpy as np
import plotly.graph_objects as go

# Parámetros físicos
M_sun = 1.98847e30
a = 1.0           # AU
m = 5.972e24 / M_sun
M = 1.0
R_SOI = a * (m/M)**(2/5)

# Centro
c_x, c_y, c_z = 1.0, 0.0, 0.0

# === Ahora solo 5 radios ===
radios = np.linspace(R_SOI, 0.5, 5)

# Paleta de colores suave
colores = ["red", "green", "purple", "orange", "cyan"]

# Malla para las esferas
theta = np.linspace(0, 2*np.pi, 60)
phi   = np.linspace(0, np.pi, 60)
theta, phi = np.meshgrid(theta, phi)

def esfera(cx, cy, cz, radius, color="blue", opacity=0.08):
    x = cx + radius * np.cos(theta) * np.sin(phi)
    y = cy + radius * np.sin(theta) * np.sin(phi)
    z = cz + radius * np.cos(phi)

    return go.Surface(
        x=x, y=y, z=z,
        opacity=opacity,
        colorscale=[[0, color], [1, color]],
        showscale=False
    )

# === FIGURA ===
fig = go.Figure()

# -------------------------
# 1. Tierra
# -------------------------
earth_radius = 4.2633689839572194e-05
fig.add_trace(esfera(c_x, c_y, c_z, earth_radius, color="blue", opacity=1.0))

# -------------------------
# 2. SOI muy transparente
# -------------------------
fig.add_trace(esfera(c_x, c_y, c_z, R_SOI, color="lightgreen", opacity=0.10))

# -------------------------
# 3. Capas esféricas: 5 radios, cada uno color distinto
# -------------------------
extra_spheres = []
for r, col in zip(radios, colores):
    extra_spheres.append(esfera(c_x, c_y, c_z, r, color=col, opacity=0.08))

# Añadirlas (ocultas por defecto)
for sp in extra_spheres:
    sp.visible = False
    fig.add_trace(sp)

# -------------------------
# Dropdown
# -------------------------
buttons = [
    dict(
        label="Ninguna",
        method="update",
        args=[{"visible": [True, True] + [False]*len(extra_spheres)}]
    )
]

for i, r in enumerate(radios):
    vis = [True, True] + [False]*len(extra_spheres)
    vis[2 + i] = True

    buttons.append(
        dict(
            label=f"Radio = {r:.3f} AU",
            method="update",
            args=[{"visible": vis}]
        )
    )

fig.update_layout(
    title="Capas esféricas alrededor de la Tierra (5 radios)",
    width=800,
    height=800,
    scene=dict(
        xaxis=dict(title='X [AU]'),
        yaxis=dict(title='Y [AU]'),
        zaxis=dict(title='Z [AU]'),
        aspectmode='data'
    ),
    updatemenus=[
        dict(
            type="dropdown",
            buttons=buttons,
            x=1.15,
            y=0.9
        )
    ]
)
fig.show()


In [1]:
import numpy as np
import plotly.graph_objects as go

# Parámetros físicos
M_sun = 1.98847e30
a = 1.0           # AU
m = 5.972e24 / M_sun
M = 1.0
R_SOI = a * (m/M)**(2/5)

# Centro
c_x, c_y, c_z = 1.0, 0.0, 0.0

# === Ahora solo 5 radios ===
radios = np.linspace(R_SOI, 0.5, 5)

# Paleta de colores suave
colores = ["red", "green", "purple", "orange", "cyan"]

# Malla para las esferas
theta = np.linspace(0, 2*np.pi, 60)
phi   = np.linspace(0, np.pi, 60)
theta, phi = np.meshgrid(theta, phi)

def esfera(cx, cy, cz, radius, color="blue", opacity=0.08):
    x = cx + radius * np.cos(theta) * np.sin(phi)
    y = cy + radius * np.sin(theta) * np.sin(phi)
    z = cz + radius * np.cos(phi)

    return go.Surface(
        x=x, y=y, z=z,
        opacity=opacity,
        colorscale=[[0, color], [1, color]],
        showscale=False
    )

# === FIGURA ===
fig = go.Figure()

# -------------------------
# 1. Tierra
# -------------------------
earth_radius = 4.2633689839572194e-05
fig.add_trace(esfera(c_x, c_y, c_z, earth_radius, color="blue", opacity=1.0))

# -------------------------
# 2. SOI muy transparente
# -------------------------
fig.add_trace(esfera(c_x, c_y, c_z, R_SOI, color="lightgreen", opacity=0.10))

# -------------------------
# 3. Capas esféricas: 5 radios, cada uno color distinto
# -------------------------
extra_spheres = []
for r, col in zip(radios, colores):
    extra_spheres.append(esfera(c_x, c_y, c_z, r, color=col, opacity=0.08))

# Añadirlas (ocultas por defecto)
for sp in extra_spheres:
    sp.visible = False
    fig.add_trace(sp)

# -------------------------
# Dropdown
# -------------------------
buttons = [
    dict(
        label="Ninguna",
        method="update",
        args=[{"visible": [True, True] + [False]*len(extra_spheres)}]
    )
]

for i, r in enumerate(radios):
    vis = [True, True] + [False]*len(extra_spheres)
    vis[2 + i] = True

    buttons.append(
        dict(
            label=f"Radio = {r:.3f} AU",
            method="update",
            args=[{"visible": vis}]
        )
    )

fig.update_layout(
    title="Capas esféricas alrededor de la Tierra (5 radios)",
    width=800,
    height=800,
    scene=dict(
        xaxis=dict(title='X [AU]'),
        yaxis=dict(title='Y [AU]'),
        zaxis=dict(title='Z [AU]'),
        aspectmode='data'
    ),
    updatemenus=[
        dict(
            type="dropdown",
            buttons=buttons,
            x=1.15,
            y=0.9
        )
    ]
)

# === ÓRBITA DE MARTE ===
r_marte = 1.5   # AU
t = np.linspace(0, 2*np.pi, 300)
x_m = r_marte * np.cos(t)
y_m = r_marte * np.sin(t)
z_m = np.zeros_like(t)

fig.add_trace(go.Scatter3d(
    x=x_m, y=y_m, z=z_m,
    mode="lines",
    line=dict(width=6, color="orange"),
    name="Órbita de Marte"
))
fig.show()


In [5]:
c_x = 1
c_y = 0
c_z = 0
v_x = 0
v_y = (mu/1)**0.5
v_z = 0

dxyz = 0.2
dvxyz = 5000 * (1/AU_m) * year 

center = (c_x, c_y, c_z, v_x, v_y, v_z)
widths = (dxyz, dxyz, dxyz, dvxyz, dvxyz, dvxyz)

In [6]:
center

(1, 0, 0, 0, 6.2840227308020005, 0)

In [8]:
widths

(0.2, 0.2, 0.2, 1.0547326203208556, 1.0547326203208556, 1.0547326203208556)

In [9]:
# resumen completo de menos de dos paginas 